# ML Coding — Day 2: Attention II (backward & training)

**~1 hour, vectorized NumPy only.** Today is **backward passes** — the harder, more interview-relevant
half. Each question gives you an upstream gradient and asks for the input gradients (the VJP), and the
test **checks your analytic gradient against a central finite-difference estimate**. No element loops
in your solutions (loops appear only inside tests / the numeric-grad helper).

Run the **test-helpers cell first** (it defines `_softmax` and `_numgrad`).

Today: **Q1** softmax backward `[warmup]` · **Q2** softmax cross-entropy fwd+bwd `[medium]` ·
**Q3** attention backward `dQ/dK/dV` `[medium]` · **Q4** causal attention fwd+bwd `[hard]` ·
**Q5** multi-head attention backward via einsum `[hard · trick]`.

---

In [ ]:
# --- test helpers (run me first) ---
import numpy as np


def _softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def _numgrad(f, x, eps=1e-6):
    """Central finite-difference gradient of scalar-valued f at array x (loops allowed in tests)."""
    x = np.asarray(x, float)
    grad = np.zeros_like(x)
    flat, gflat = x.reshape(-1), grad.reshape(-1)      # views: mutating flat mutates x
    for i in range(flat.size):
        o = flat[i]
        flat[i] = o + eps; a = f(x)
        flat[i] = o - eps; b = f(x)
        flat[i] = o
        gflat[i] = (a - b) / (2 * eps)
    return grad

## Q1 — Softmax backward (VJP)  ·  `[warmup]`

**Why it matters:** softmax sits inside attention, cross-entropy, and MoE routing — its backward is
the building block for all of today. The Jacobian `∂p_i/∂x_j = p_i(δ_ij − p_j)` is dense `(C×C)`, but
the vector-Jacobian product collapses to an **O(C)** formula, so you never materialize it.

Implement `softmax_backward(g, p) -> dx`: given upstream `g = ∂L/∂p` and the softmax **output** `p`
(both `(..., C)`, softmax over the last axis), return `∂L/∂x`.

**Hint:** `dx = p ⊙ (g − Σ_k g_k p_k)` with the sum over the last axis (`keepdims=True`). No loops.

In [ ]:
import numpy as np


def softmax_backward(g, p):
    raise NotImplementedError

In [ ]:
# --- Q1 tests ---
def _q1():
    rng = np.random.default_rng(0)
    x = rng.standard_normal((3, 5))
    g = rng.standard_normal((3, 5))         # arbitrary upstream gradient
    p = _softmax(x)
    dx = softmax_backward(g, p)
    assert dx.shape == x.shape
    num = _numgrad(lambda z: np.sum(g * _softmax(z)), x)
    assert np.allclose(dx, num, atol=1e-6)
    # sanity: gradient of (sum of probs == 1) is 0 -> g constant gives dx == 0
    assert np.allclose(softmax_backward(np.ones_like(p), p), 0.0, atol=1e-12)
    print("Q1 OK")

_q1()

## Q2 — Softmax cross-entropy, fused forward + backward  ·  `[medium]`

**Why it matters:** the single most common training loss, and the textbook example of why you fuse
softmax with the loss: the gradient collapses to the famous **`∂L/∂logits = (p − onehot(y)) / N`** —
no dense softmax Jacobian, and computing the loss via log-sum-exp avoids ever forming tiny
probabilities.

Implement `softmax_ce(logits, labels) -> (loss, dlogits)`:
- `logits` `(N, C)`, integer `labels` `(N,)`; `loss` = mean negative log-likelihood of the true class.
- Compute `loss` stably (log-sum-exp) and return `dlogits` `(N, C)`.

**Hint:** `logp = logits − logsumexp(logits)`; `loss = −mean(logp[n, y_n])`; `dlogits = softmax(logits)`
with `1/N` subtracted at the true-class entries (fancy indexing). No loops.

In [ ]:
def softmax_ce(logits, labels):
    raise NotImplementedError

In [ ]:
# --- Q2 tests ---
def _q2():
    rng = np.random.default_rng(1)
    N, C = 4, 5
    logits = rng.standard_normal((N, C))
    labels = rng.integers(0, C, N)
    loss, d = softmax_ce(logits, labels)
    assert np.isscalar(loss) or loss.ndim == 0
    assert d.shape == (N, C)
    p = _softmax(logits)
    assert abs(loss - (-np.mean(np.log(p[np.arange(N), labels])))) < 1e-9    # independent loss check
    num = _numgrad(lambda x: softmax_ce(x, labels)[0], logits)
    assert np.allclose(d, num, atol=1e-6)
    big = np.array([[0.0, 0.0, 50.0]])                                       # confident & correct
    assert softmax_ce(big, np.array([2]))[0] < 1e-6
    print("Q2 OK")

_q2()

## Q3 — Scaled dot-product attention backward  ·  `[medium]`

**Paper:** Vaswani et al., 2017 (arXiv:1706.03762). **Why it matters:** training a transformer means
backpropagating through attention. The chain is short but every step is a batched matmul, and it
reuses the softmax VJP from Q1.

Forward (given): `S = Q·Kᵀ/√d`, `P = softmax(S)`, `O = P·V`, with `Q (B,Tq,d)`, `K (B,Tk,d)`,
`V (B,Tk,dv)`. Implement `attention_backward(dO, Q, K, V) -> (dQ, dK, dV)` for upstream `dO (B,Tq,dv)`.

**Hint (derive these):** `dV = Pᵀ·dO`; `dP = dO·Vᵀ`; `dS = softmax_backward(dP, P)`;
`dQ = dS·K / √d`; `dK = dSᵀ·Q / √d`. Use `np.swapaxes(·, -1, -2)` for the batched transposes. No loops.

In [ ]:
def attention_backward(dO, Q, K, V):
    raise NotImplementedError

In [ ]:
# --- Q3 tests ---
def _q3():
    rng = np.random.default_rng(2)
    B, Tq, Tk, d, dv = 2, 3, 4, 5, 6
    Q = rng.standard_normal((B, Tq, d)); K = rng.standard_normal((B, Tk, d)); V = rng.standard_normal((B, Tk, dv))
    g = rng.standard_normal((B, Tq, dv))    # upstream dO

    def fwd(q, k, v):
        s = q @ np.swapaxes(k, -1, -2) / np.sqrt(q.shape[-1])
        return _softmax(s) @ v

    dQ, dK, dV = attention_backward(g, Q, K, V)
    assert dQ.shape == Q.shape and dK.shape == K.shape and dV.shape == V.shape
    assert np.allclose(dQ, _numgrad(lambda q: np.sum(g * fwd(q, K, V)), Q), atol=1e-5)
    assert np.allclose(dK, _numgrad(lambda k: np.sum(g * fwd(Q, k, V)), K), atol=1e-5)
    assert np.allclose(dV, _numgrad(lambda v: np.sum(g * fwd(Q, K, v)), V), atol=1e-5)
    print("Q3 OK")

_q3()

## Q4 — Causal attention, forward + backward  ·  `[hard]`

**Paper:** Vaswani et al., 2017. **Why it matters:** the causal mask makes attention autoregressive
(GPT-style). The subtle part is the **backward**: masked positions must receive **exactly zero**
gradient. They do automatically — `P = 0` there, so `dS = P ⊙ (…) = 0` — but a naive implementation
that adds gradient on masked entries is a classic bug, which the finite-diff check will catch.

Self-attention with `Q, K (B, T, d)`, `V (B, T, dv)`. Implement both:
- `causal_attention(Q, K, V) -> O` — mask scores with a lower-triangular mask (`-inf` above the
  diagonal) before softmax.
- `causal_attention_backward(dO, Q, K, V) -> (dQ, dK, dV)`.

**Hint:** same VJPs as Q3, but compute `P` from the *masked* scores; everything else follows. No loops.

In [ ]:
def causal_attention(Q, K, V):
    raise NotImplementedError


def causal_attention_backward(dO, Q, K, V):
    raise NotImplementedError

In [ ]:
# --- Q4 tests ---
def _q4():
    rng = np.random.default_rng(3)
    B, T, d, dv = 2, 4, 5, 6
    Q = rng.standard_normal((B, T, d)); K = rng.standard_normal((B, T, d)); V = rng.standard_normal((B, T, dv))
    O = causal_attention(Q, K, V)
    assert O.shape == (B, T, dv)
    V2 = V.copy(); V2[:, 1:] += 100.0                       # future tokens
    assert np.allclose(O[:, 0], causal_attention(Q, K, V2)[:, 0])   # token 0 sees only itself

    g = rng.standard_normal((B, T, dv))
    dQ, dK, dV = causal_attention_backward(g, Q, K, V)
    assert np.allclose(dQ, _numgrad(lambda q: np.sum(g * causal_attention(q, K, V)), Q), atol=1e-5)
    assert np.allclose(dK, _numgrad(lambda k: np.sum(g * causal_attention(Q, k, V)), K), atol=1e-5)
    assert np.allclose(dV, _numgrad(lambda v: np.sum(g * causal_attention(Q, K, v)), V), atol=1e-5)
    print("Q4 OK")

_q4()

## Q5 — Multi-head attention backward via einsum  ·  `[hard · tensor trick]`

**Paper:** Vaswani et al., 2017. **Why it matters:** the full MHA block backward is what an optimizer
actually runs. The **trick** is doing all the per-head, batched contractions with `np.einsum` and
getting the weight gradients as outer-product contractions — plus reshaping/​transposing heads
correctly on the way *down* (the VJP of `split` is `merge`).

Block: `Q=X·Wq, K=X·Wk, V=X·Wv` (`X (B,T,D)`, each `W (D,D)`), split into `H` heads `(B,H,T,d)`,
full (non-causal) attention per head, concat to `(B,T,D)`, then `out = ctx·Wo`. Implement:
- `mha_forward(X, Wq, Wk, Wv, Wo, num_heads) -> out`
- `mha_backward(dOut, X, Wq, Wk, Wv, Wo, num_heads) -> (dX, dWq, dWk, dWv, dWo)`

**Hint:** `dWo = einsum('bti,btj->ij', ctx, dOut)`, `dctx = dOut·Woᵀ`; split `dctx` into heads;
`dVh = einsum('bhqk,bhqd->bhkd', P, dctxh)`, `dP = einsum('bhqd,bhkd->bhqk', dctxh, Vh)`,
`dS = softmax_backward(dP, P)`, `dQh = einsum('bhqk,bhkd->bhqd', dS, Kh)/√d`,
`dKh = einsum('bhqk,bhqd->bhkd', dS, Qh)/√d`; merge heads back, then
`dWq = einsum('bti,btj->ij', X, dQ)` and `dX = dQ·Wqᵀ + dK·Wkᵀ + dV·Wvᵀ`. No loops.

In [ ]:
def mha_forward(X, Wq, Wk, Wv, Wo, num_heads):
    raise NotImplementedError


def mha_backward(dOut, X, Wq, Wk, Wv, Wo, num_heads):
    raise NotImplementedError

In [ ]:
# --- Q5 tests ---
def _q5():
    rng = np.random.default_rng(4)
    B, T, D, H = 2, 3, 4, 2
    X = rng.standard_normal((B, T, D))
    Wq, Wk, Wv, Wo = (rng.standard_normal((D, D)) for _ in range(4))
    out = mha_forward(X, Wq, Wk, Wv, Wo, H)
    assert out.shape == (B, T, D)

    g = rng.standard_normal((B, T, D))
    dX, dWq, dWk, dWv, dWo = mha_backward(g, X, Wq, Wk, Wv, Wo, H)
    assert dX.shape == X.shape and dWq.shape == Wq.shape and dWo.shape == Wo.shape

    def L(Xv, q, k, v, o):
        return np.sum(g * mha_forward(Xv, q, k, v, o, H))

    assert np.allclose(dX, _numgrad(lambda a: L(a, Wq, Wk, Wv, Wo), X), atol=1e-5)
    assert np.allclose(dWq, _numgrad(lambda a: L(X, a, Wk, Wv, Wo), Wq), atol=1e-5)
    assert np.allclose(dWk, _numgrad(lambda a: L(X, Wq, a, Wv, Wo), Wk), atol=1e-5)
    assert np.allclose(dWv, _numgrad(lambda a: L(X, Wq, Wk, a, Wo), Wv), atol=1e-5)
    assert np.allclose(dWo, _numgrad(lambda a: L(X, Wq, Wk, Wv, a), Wo), atol=1e-5)
    print("Q5 OK")

_q5()

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import numpy as np


def _sm(z):                                            # stable softmax over last axis
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


def ref_softmax_backward(g, p):
    g = np.asarray(g, float); p = np.asarray(p, float)
    return p * (g - (g * p).sum(axis=-1, keepdims=True))


def ref_softmax_ce(logits, labels):
    logits = np.asarray(logits, float); labels = np.asarray(labels)
    N = logits.shape[0]
    z = logits - logits.max(axis=1, keepdims=True)
    logZ = np.log(np.exp(z).sum(axis=1, keepdims=True))
    logp = z - logZ
    loss = -logp[np.arange(N), labels].mean()
    d = np.exp(logp)                                   # = softmax(logits)
    d[np.arange(N), labels] -= 1.0
    d /= N
    return loss, d


def ref_attention_backward(dO, Q, K, V):
    Q = np.asarray(Q, float); K = np.asarray(K, float); V = np.asarray(V, float); dO = np.asarray(dO, float)
    d = Q.shape[-1]
    P = _sm(Q @ np.swapaxes(K, -1, -2) / np.sqrt(d))
    dV = np.swapaxes(P, -1, -2) @ dO
    dP = dO @ np.swapaxes(V, -1, -2)
    dS = P * (dP - (dP * P).sum(axis=-1, keepdims=True))
    dQ = dS @ K / np.sqrt(d)
    dK = np.swapaxes(dS, -1, -2) @ Q / np.sqrt(d)
    return dQ, dK, dV


def _causal_P(Q, K):
    d = Q.shape[-1]; T = Q.shape[-2]
    s = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d)
    s = np.where(np.tril(np.ones((T, T), bool)), s, -np.inf)
    return _sm(s)


def ref_causal_attention(Q, K, V):
    Q = np.asarray(Q, float); K = np.asarray(K, float); V = np.asarray(V, float)
    return _causal_P(Q, K) @ V


def ref_causal_attention_backward(dO, Q, K, V):
    Q = np.asarray(Q, float); K = np.asarray(K, float); V = np.asarray(V, float); dO = np.asarray(dO, float)
    d = Q.shape[-1]
    P = _causal_P(Q, K)                                # masked weights (zeros above diagonal)
    dV = np.swapaxes(P, -1, -2) @ dO
    dP = dO @ np.swapaxes(V, -1, -2)
    dS = P * (dP - (dP * P).sum(axis=-1, keepdims=True))   # zero where P==0, so no leak
    dQ = dS @ K / np.sqrt(d)
    dK = np.swapaxes(dS, -1, -2) @ Q / np.sqrt(d)
    return dQ, dK, dV


def ref_mha_forward(X, Wq, Wk, Wv, Wo, num_heads):
    X = np.asarray(X, float)
    B, T, D = X.shape; H = num_heads; d = D // H
    sp = lambda M: M.reshape(B, T, H, d).transpose(0, 2, 1, 3)      # (B,T,D) -> (B,H,T,d)
    Qh, Kh, Vh = sp(X @ Wq), sp(X @ Wk), sp(X @ Wv)
    P = _sm(np.einsum("bhqd,bhkd->bhqk", Qh, Kh) / np.sqrt(d))
    ctxh = np.einsum("bhqk,bhkd->bhqd", P, Vh)
    ctx = ctxh.transpose(0, 2, 1, 3).reshape(B, T, D)              # concat heads
    return ctx @ Wo


def ref_mha_backward(dOut, X, Wq, Wk, Wv, Wo, num_heads):
    X = np.asarray(X, float); dOut = np.asarray(dOut, float)
    B, T, D = X.shape; H = num_heads; d = D // H
    sp = lambda M: M.reshape(B, T, H, d).transpose(0, 2, 1, 3)
    mg = lambda Mh: Mh.transpose(0, 2, 1, 3).reshape(B, T, D)
    Qh, Kh, Vh = sp(X @ Wq), sp(X @ Wk), sp(X @ Wv)
    P = _sm(np.einsum("bhqd,bhkd->bhqk", Qh, Kh) / np.sqrt(d))
    ctxh = np.einsum("bhqk,bhkd->bhqd", P, Vh)
    ctx = mg(ctxh)
    dWo = np.einsum("bti,btj->ij", ctx, dOut)
    dctxh = sp(dOut @ Wo.T)
    dVh = np.einsum("bhqk,bhqd->bhkd", P, dctxh)
    dP = np.einsum("bhqd,bhkd->bhqk", dctxh, Vh)
    dS = P * (dP - (dP * P).sum(axis=-1, keepdims=True))
    dQh = np.einsum("bhqk,bhkd->bhqd", dS, Kh) / np.sqrt(d)
    dKh = np.einsum("bhqk,bhqd->bhkd", dS, Qh) / np.sqrt(d)
    dQ, dK, dV = mg(dQh), mg(dKh), mg(dVh)
    dWq = np.einsum("bti,btj->ij", X, dQ)
    dWk = np.einsum("bti,btj->ij", X, dK)
    dWv = np.einsum("bti,btj->ij", X, dV)
    dX = dQ @ Wq.T + dK @ Wk.T + dV @ Wv.T
    return dX, dWq, dWk, dWv, dWo


# sanity-check the references
_p = _sm(np.array([[1., 2., 3.]]))
assert np.allclose(ref_softmax_backward(np.ones_like(_p), _p), 0.0, atol=1e-12)
_loss, _d = ref_softmax_ce(np.array([[0., 0., 50.]]), np.array([2]))
assert _loss < 1e-6 and _d.shape == (1, 3)
# attention_backward: dO (1,Tq,dv), Q (1,Tq,d), K (1,Tk,d), V (1,Tk,dv) -> dQ has Q's shape
assert ref_attention_backward(np.ones((1, 3, 4)), np.zeros((1, 3, 3)),
                              np.zeros((1, 2, 3)), np.ones((1, 2, 4)))[0].shape == (1, 3, 3)
assert ref_causal_attention(np.zeros((1, 3, 2)), np.zeros((1, 3, 2)), np.arange(6.).reshape(1, 3, 2)).shape == (1, 3, 2)
assert ref_mha_forward(np.zeros((1, 3, 4)), *[np.eye(4)] * 4, 2).shape == (1, 3, 4)
print("reference solutions OK")